In [2]:
import SimpleITK as sitk
import numpy as np

# --- Input files ---
b1_file = "/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/missing_qualitative_imaging/missing everything/CE2-2301-115/P115_dwi_b1.nrrd"   # image with lower b-value (e.g. b=0)
b2_file = "/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/missing_qualitative_imaging/missing everything/CE2-2301-115/P115_dwi_b2.nrrd"  # image with higher b-value (e.g. b=1000)
b1 = 50
b2 = 900

# --- Load NRRD images ---
img_b1 = sitk.ReadImage(b1_file)
img_b2 = sitk.ReadImage(b2_file)

# Convert to numpy arrays
S1 = sitk.GetArrayFromImage(img_b1).astype(np.float32)
S2 = sitk.GetArrayFromImage(img_b2).astype(np.float32)

# Avoid division by zero
S1[S1 <= 0] = 1e-6
S2[S2 <= 0] = 1e-6

#S2=S2[1:,:,:].    # sometimes need to adjust sizes

# --- Compute ADC map ---
adc = 1000000*np.log(S1 / S2) / (b2 - b1)

# Remove negative or extreme values
adc[adc < 0] = 0
adc[adc > 15000] = 0.00  # typical upper limit for brain DWI

# --- Save ADC as NRRD ---
adc_img = sitk.GetImageFromArray(adc)
adc_img.CopyInformation(img_b1)  # preserve spacing/origin/direction
sitk.WriteImage(adc_img, "/Volumes/T7 Shield/Restrepo/HCC_Retrospective_radiomics/missing_qualitative_imaging/missing everything/CE2-2301-115/P115_adc.nrrd")

In [14]:
S2.shape

(30, 256, 256)